# 03. Тестовая выборка и итоговые сравнения
Для генерации требуется GPU. Имя запуска, признак и каталог результатов должны совпадать с настройками ноутбука 01.

In [ ]:
from hashlib import sha256
from pathlib import Path
from runpy import run_path
from subprocess import run

REPO_URL = "https://github.com/danilov-e/lora-activation-steering.git"
RUNS_DIR = Path("/content/drive/MyDrive/lora_activation_steering/runs")
RUN_NAME = "research_01"
TRAIT = "sycophancy"

repo_id = sha256(REPO_URL.encode()).hexdigest()[:12]
source = Path("/content/lora-steering") / repo_id / "source"
if not source.exists():
    source.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(source)], check=True)
else:
    run(["git", "-C", str(source), "fetch", "--quiet", "origin", "HEAD"], check=True)
    run(["git", "-C", str(source), "checkout", "--quiet", "--detach", "FETCH_HEAD"], check=True)

colab = run_path(str(source / "notebooks" / "colab.py"))
PROJECT, RUNS, _ = colab["setup"](source, RUNS_DIR, RUN_NAME, TRAIT, resume=True)

from src import judge as j, report as r, workflow as w
from IPython.display import Image, display
import pandas as pd

In [ ]:
RUN = w.open_run(PROJECT, RUN_NAME, TRAIT, runs_dir=RUNS)
CONFIG = w.run_info(RUN)["config"]
choices = w.load_choices(RUN)
print(RUN)
display(pd.DataFrame(choices))

TEST = RUN / "test" / "all"
expected = (2 + 8 * len(choices)) * len(w.pairs(RUN, "test")) * CONFIG["test_repeats"]
batch_size = CONFIG["judge"]["batch_size"]
print(f"Test: {expected} ответов, около {(expected + batch_size - 1) // batch_size} запросов")

## 1. Генерация ответов
До генерации проверяется наличие итоговых адаптеров и векторов. При повторном выполнении сохранённые ответы используются повторно.

In [ ]:
w.generate_test(RUN)

## 2. Оценка через OpenAI
Ячейка выполняет платные запросы. Модель судьи и уровень рассуждения берутся из настроек запуска, общих для валидационной и тестовой выборок.

In [ ]:
colab["api_key"]()
display(j.judge(TEST, TRAIT, **CONFIG["judge"]))

## 3. Итоговые сравнения
Отчёт содержит эффект стиринга относительно базового условия каждой модели и разницу с исходной моделью под исходным вектором. Для парных доверительных интервалов повторы сначала усредняются по вопросу.

In [ ]:
frame = r.scored(TEST, TRAIT)
summary, effects, figures = r.test_report(frame, RUN)
display(summary.round(2))
display(effects.round(2))
for plot in figures:
    display(Image(filename=str(plot)))

In [ ]:
for choice in choices:
    name = choice["name"]
    print(name)
    display(r.examples(frame, TRAIT, f"steered_{name}:{name}_old"))